<a href="https://colab.research.google.com/github/dalkhaseeb-D/at-risk-student-prediction/blob/main/OULAD_Baseline_Student_Grouped.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/oulad_baseline_review")

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/dalkhaseeb-D/at-risk-student-prediction.git",
            str(PROJECT_DIR),
        ],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

print("GitHub version:", commit)
print("\nPython files:")
for path in sorted(PROJECT_DIR.rglob("*.py")):
    if ".git" not in path.parts:
        print(path.relative_to(PROJECT_DIR))

print("\nStep 1 completed.")

GitHub version: 9ce9278542c89ec878d2532eb7550e1aa39719be

Python files:
make_figures.py
run_experiment.py

Step 1 completed.


In [2]:
import sys
import subprocess

requirements = PROJECT_DIR / "requirements.txt"

print("Installing project libraries...", flush=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-r", str(requirements)
    ],
    check=True
)

print("\nStep 2 completed: libraries installed.")

Installing project libraries...

Step 2 completed: libraries installed.


In [3]:
import sys
import importlib
import json

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

original = importlib.import_module("run_experiment")

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "student_grouped_results"
FIGURES_DIR = PROJECT_DIR / "student_grouped_figures"

for folder in [DATA_DIR, RESULTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("1. Downloading OULAD data...", flush=True)
original.download_oulad(DATA_DIR)

print("\n2. Auditing the data...", flush=True)
tables = original.audit(DATA_DIR, RESULTS_DIR)

print("\n3. Building the original day-28 dataset...", flush=True)
model_data = original.build_dataset(
    tables, DATA_DIR, RESULTS_DIR
)

# At prediction time, the registration must already have started.
late_registration = model_data["date_registration"].gt(28)

print(
    "\nRegistrations starting after day 28:",
    int(late_registration.sum())
)

model_data = (
    model_data.loc[~late_registration]
    .reset_index(drop=True)
)

print("\nCorrected registrations:", len(model_data))
print("Unique students:", model_data["id_student"].nunique())
print(
    "At-risk percentage:",
    round(model_data["target_at_risk"].mean() * 100, 2)
)

model_data.to_csv(
    RESULTS_DIR / "day28_model_data.csv",
    index=False
)

print("\nStep 3 completed: data ready; no models trained.")

1. Downloading OULAD data...
[data] downloading OULAD from https://archive.ics.uci.edu/static/public/349/open%2Buniversity%2Blearning%2Banalytics%2Bdataset.zip

2. Auditing the data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                   File     Rows  Columns  Duplicate_Rows  NaN_naive_audit  Missing_as_?
            courses.csv       22        3               0                0             0
        assessments.csv      206        6               0                0            11
        studentInfo.csv    32593       12               0                0          1111
studentRegistration.csv    32593        5               0                0         22566
  studentAssessment.csv   173912        5               0                0           173
                vle.csv     6364        6               0                0         10486
         studentVle.csv 10655280        6          787170                0             0
               File              Column  Missing
        assessments                date       11
        studentInfo            imd_band     1111
studentRegistration   date_registration       45
studentRegistration date_unregistration    22521
  studentAssessment               score    

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "registrations": 32593,
  "withdrew_before_day28_excluded": 5055,
  "eligible_at_day28": 27538,
  "at_risk": 12153,
  "at_risk_pct": 44.13,
  "no_vle_activity_wk1_4": 1246,
  "unique_students": 24839,
  "students_with_multiple_registrations": 2578,
  "final_result_counts": {
    "Pass": 12361,
    "Fail": 7044,
    "Withdrawn": 5109,
    "Distinction": 3024
  }
}

Registrations starting after day 28: 16

Corrected registrations: 27522
Unique students: 24832
At-risk percentage: 44.12

Step 3 completed: data ready; no models trained.


In [4]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

categorical_features = original.CATEGORICAL
numeric_features = original.PROFILE_NUMERIC + original.ENGAGEMENT

X = model_data[categorical_features + numeric_features].copy()
y = model_data["target_at_risk"].copy()
groups = model_data["id_student"].copy()

assert groups.notna().all(), "Some student IDs are missing."

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Use the first fold, fixed before training or viewing performance.
train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_students = set(groups.iloc[train_idx])
test_students = set(groups.iloc[test_idx])
overlap = train_students & test_students

assert len(overlap) == 0, "Student overlap detected!"

split_summary = pd.DataFrame([
    {
        "Partition": "Train",
        "Registrations": len(train_idx),
        "Unique students": len(train_students),
        "At-risk %": round(y_train.mean() * 100, 2)
    },
    {
        "Partition": "Test",
        "Registrations": len(test_idx),
        "Unique students": len(test_students),
        "At-risk %": round(y_test.mean() * 100, 2)
    }
])

display(split_summary)
print("Students shared between train and test:", len(overlap))

assignments = model_data[original.KEYS].copy()
assignments["partition"] = ""
assignments.loc[X_train.index, "partition"] = "train"
assignments.loc[X_test.index, "partition"] = "test"

assignments.to_csv(
    RESULTS_DIR / "student_split_assignments.csv",
    index=False
)

split_summary.to_csv(
    RESULTS_DIR / "student_split_summary.csv",
    index=False
)

print("\nStep 4 completed: student-separated split saved.")

,Partition,Registrations,Unique students,At-risk %
0,Train,22017,19865,44.12
1,Test,5505,4967,44.12


Students shared between train and test: 0

Step 4 completed: student-separated split saved.


In [5]:
import numpy as np
import pandas as pd

print("Training the original baseline models...", flush=True)

model_results, fitted_models = original.fit_and_score(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    categorical=categorical_features,
    numeric=numeric_features
)

model_results = (
    model_results
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

model_results.to_csv(
    RESULTS_DIR / "04_model_comparison.csv",
    index=False
)

# Same fixed ensemble as the original project.
ensemble_names = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting"
]

ensemble_probabilities = np.mean(
    [
        fitted_models[name].predict_proba(X_test)[:, 1]
        for name in ensemble_names
    ],
    axis=0
)

ensemble_predictions = (
    ensemble_probabilities >= 0.5
).astype(int)

ensemble_result = pd.DataFrame([{
    "Model": "Soft Voting Ensemble (LR+RF+GB)",
    **original.scores(
        y_test,
        ensemble_predictions,
        ensemble_probabilities
    )
}])

ensemble_result.to_csv(
    RESULTS_DIR / "04_ensemble.csv",
    index=False
)

print("\nIndividual models:")
display(model_results.round(3))

print("\nFixed soft-voting ensemble:")
display(ensemble_result.round(3))

# Save test predictions for reproducibility.
test_predictions = model_data.loc[
    X_test.index,
    original.KEYS + ["target_at_risk"]
].copy()

for name, pipeline in fitted_models.items():
    test_predictions[f"{name}_probability"] = (
        pipeline.predict_proba(X_test)[:, 1]
    )

test_predictions["ensemble_probability"] = ensemble_probabilities

test_predictions.to_csv(
    RESULTS_DIR / "04_test_predictions.csv",
    index=False
)

print("\nStep 5 completed: models evaluated and results saved.")

Training the original baseline models...

Individual models:


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,Train time (s),Parameters / nodes,Size (KB)
0,Logistic Regression,0.679,0.678,0.628,0.671,0.648,0.744,0.383,53,1.108
1,Gradient Boosting,0.683,0.678,0.640,0.641,0.641,0.747,1.073,5734,337.850
2,Random Forest,0.680,0.674,0.642,0.621,0.631,0.740,2.407,242534,19011.212
3,Decision Tree,0.665,0.655,0.637,0.563,0.597,0.712,0.182,319,26.053
4,Dummy Baseline,0.559,0.500,0.000,0.000,0.000,0.500,0.105,2,0.466



Fixed soft-voting ensemble:


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC
0,Soft Voting Ensemble (LR+RF+GB),0.683,0.679,0.638,0.648,0.643,0.749



Step 5 completed: models evaluated and results saved.


In [6]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

train_groups = model_data.loc[
    X_train.index, "id_student"
]

grouped_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Fix and audit all folds before training.
cv_splits = list(
    grouped_cv.split(
        X_train,
        y_train,
        groups=train_groups
    )
)

fold_audit = []

for fold, (fit_idx, val_idx) in enumerate(cv_splits, start=1):
    fit_students = set(train_groups.iloc[fit_idx])
    val_students = set(train_groups.iloc[val_idx])

    shared = len(fit_students & val_students)
    assert shared == 0

    fold_audit.append({
        "Fold": fold,
        "Training registrations": len(fit_idx),
        "Validation registrations": len(val_idx),
        "Student overlap": shared
    })

fold_audit = pd.DataFrame(fold_audit)
display(fold_audit)

fold_audit.to_csv(
    RESULTS_DIR / "06_grouped_cv_fold_audit.csv",
    index=False
)

scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_rows = []

for name, model in original.make_models().items():
    if name == "Dummy Baseline":
        continue

    print("Grouped CV:", name, flush=True)

    pipeline = Pipeline([
        (
            "pre",
            original.make_preprocessor(
                categorical_features,
                numeric_features
            )
        ),
        ("model", model)
    ])

    result = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_splits,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1
    )

    for metric in scoring:
        cv_rows.append({
            "Model": name,
            "Metric": metric,
            "Train mean": result[f"train_{metric}"].mean(),
            "Validation mean": result[f"test_{metric}"].mean(),
            "Validation SD": result[f"test_{metric}"].std(ddof=1)
        })

cv_summary = pd.DataFrame(cv_rows)

cv_summary.to_csv(
    RESULTS_DIR / "06_grouped_cross_validation.csv",
    index=False
)

display(cv_summary.round(3))

print("\nStep 6 completed: grouped CV on training data only.")

,Fold,Training registrations,Validation registrations,Student overlap
0,1,17614,4403,0
1,2,17614,4403,0
2,3,17614,4403,0
3,4,17613,4404,0
4,5,17613,4404,0


Grouped CV: Logistic Regression
Grouped CV: Decision Tree
Grouped CV: Random Forest
Grouped CV: Gradient Boosting


,Model,Metric,Train mean,Validation mean,Validation SD
0,Logistic Regression,balanced_accuracy,0.680,0.678,0.008
1,Logistic Regression,precision,0.628,0.626,0.010
2,Logistic Regression,recall,0.676,0.674,0.011
3,Logistic Regression,f1,0.651,0.649,0.009
4,Logistic Regression,roc_auc,0.746,0.742,0.008
5,Decision Tree,balanced_accuracy,0.682,0.649,0.007
6,Decision Tree,precision,0.644,0.608,0.014
7,Decision Tree,recall,0.648,0.610,0.025
8,Decision Tree,f1,0.645,0.608,0.009
9,Decision Tree,roc_auc,0.753,0.708,0.009



Step 6 completed: grouped CV on training data only.


In [7]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
import pandas as pd

feature_sets = {
    "Profile only": (
        original.CATEGORICAL,
        original.PROFILE_NUMERIC
    ),
    "Engagement only": (
        [],
        original.ENGAGEMENT
    ),
    "Profile + engagement": (
        categorical_features,
        numeric_features
    )
}

ablation_rows = []

for feature_set, (cats, nums) in feature_sets.items():
    for name, model in original.make_models().items():

        if name == "Dummy Baseline":
            continue

        print(feature_set, "|", name, flush=True)

        pipeline = Pipeline([
            (
                "pre",
                original.make_preprocessor(cats, nums)
            ),
            ("model", model)
        ])

        result = cross_validate(
            pipeline,
            X_train[cats + nums],
            y_train,
            cv=cv_splits,
            scoring={"f1": "f1", "roc_auc": "roc_auc"},
            n_jobs=1
        )

        ablation_rows.append({
            "Feature set": feature_set,
            "Model": name,
            "F1 mean": result["test_f1"].mean(),
            "F1 SD": result["test_f1"].std(ddof=1),
            "AUC mean": result["test_roc_auc"].mean(),
            "AUC SD": result["test_roc_auc"].std(ddof=1)
        })

ablation_results = pd.DataFrame(ablation_rows)

ablation_results.to_csv(
    RESULTS_DIR / "07_ablation_grouped_cv.csv",
    index=False
)

display(ablation_results.round(3))

print("\nStep 7 completed: feature ablation saved.")

Profile only | Logistic Regression
Profile only | Decision Tree
Profile only | Random Forest
Profile only | Gradient Boosting
Engagement only | Logistic Regression
Engagement only | Decision Tree
Engagement only | Random Forest
Engagement only | Gradient Boosting
Profile + engagement | Logistic Regression
Profile + engagement | Decision Tree
Profile + engagement | Random Forest
Profile + engagement | Gradient Boosting


,Feature set,Model,F1 mean,F1 SD,AUC mean,AUC SD
0,Profile only,Logistic Regression,0.579,0.008,0.654,0.008
1,Profile only,Decision Tree,0.560,0.006,0.628,0.007
2,Profile only,Random Forest,0.570,0.006,0.652,0.007
3,Profile only,Gradient Boosting,0.576,0.007,0.651,0.005
4,Engagement only,Logistic Regression,0.609,0.007,0.671,0.007
5,Engagement only,Decision Tree,0.594,0.002,0.658,0.007
6,Engagement only,Random Forest,0.582,0.003,0.663,0.007
7,Engagement only,Gradient Boosting,0.594,0.007,0.668,0.008
8,Profile + engagement,Logistic Regression,0.649,0.009,0.742,0.008
9,Profile + engagement,Decision Tree,0.608,0.009,0.708,0.009



Step 7 completed: feature ablation saved.


In [8]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Held-out test results
comparison = pd.concat(
    [
        model_results[
            ["Model", "Precision", "Recall", "F1", "ROC-AUC"]
        ],
        ensemble_result[
            ["Model", "Precision", "Recall", "F1", "ROC-AUC"]
        ]
    ],
    ignore_index=True
)

ax = comparison.set_index("Model").plot.bar(
    figsize=(12, 6),
    width=0.8
)

ax.set_title("Baseline Models: Student-Separated Test Set")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "01_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

# 2. Feature-group comparison using grouped CV
feature_order = [
    "Profile only",
    "Engagement only",
    "Profile + engagement"
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, sd_column, title in [
    (axes[0], "F1 mean", "F1 SD", "At-Risk F1"),
    (axes[1], "AUC mean", "AUC SD", "ROC-AUC")
]:
    means = ablation_results.pivot(
        index="Feature set",
        columns="Model",
        values=metric
    ).reindex(feature_order)

    deviations = ablation_results.pivot(
        index="Feature set",
        columns="Model",
        values=sd_column
    ).reindex(
        index=feature_order,
        columns=means.columns
    )

    means.plot.bar(
        ax=ax,
        yerr=deviations,
        capsize=3,
        width=0.8
    )

    ax.set_title(title)
    ax.set_ylabel("Score")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.25)

fig.suptitle(
    "Feature Ablation: Five-Fold Student-Grouped CV\n"
    "Training partition only; error bars show fold SD"
)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "02_feature_ablation.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

# 3. Training vs validation AUC
auc_summary = (
    cv_summary.loc[cv_summary["Metric"] == "roc_auc"]
    .set_index("Model")
)

positions = np.arange(len(auc_summary))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    positions - width / 2,
    auc_summary["Train mean"],
    width,
    label="Training"
)

ax.bar(
    positions + width / 2,
    auc_summary["Validation mean"],
    width,
    yerr=auc_summary["Validation SD"],
    capsize=4,
    label="Grouped validation"
)

ax.set_xticks(positions)
ax.set_xticklabels(auc_summary.index, rotation=15)
ax.set_ylim(0, 1)
ax.set_ylabel("ROC-AUC")
ax.set_title("Training vs Student-Grouped Validation AUC")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "03_train_validation_auc.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

print("Step 8 completed. Saved figures:")
for path in sorted(FIGURES_DIR.glob("*.png")):
    print(path.name)

Step 8 completed. Saved figures:
01_model_comparison.png
02_feature_ablation.png
03_train_validation_auc.png


In [9]:
import json
import platform
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import sklearn
import pandas as pd
from google.colab import files

# Record the exact source version and current experiment.
metadata = {
    "source_repository":
        "https://github.com/dalkhaseeb-D/at-risk-student-prediction",
    "source_commit": subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
        text=True
    ).strip(),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
    "registrations": len(model_data),
    "unique_students": int(model_data["id_student"].nunique()),
    "at_risk_percentage": float(
        model_data["target_at_risk"].mean() * 100
    ),
    "features": list(X.columns),
    "train_registrations": len(X_train),
    "test_registrations": len(X_test),
    "student_overlap": len(overlap),
    "main_split": "First fold of shuffled StratifiedGroupKFold",
    "seed": 42,
    "cv": "5-fold student-grouped CV on training partition only",
    "classification_threshold": 0.5,
    "cohort_correction":
        "Exclude registrations starting after day 28; "
        "retain missing registration dates",
    "note":
        "02_dataset_summary.json describes the original cohort "
        "before the late-registration correction. "
        "This metadata describes the corrected run."
}

(RESULTS_DIR / "corrected_run_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

checkpoint = Path("/content/OULAD_Baseline_Grouped_Checkpoint.zip")

with ZipFile(checkpoint, "w", ZIP_DEFLATED) as archive:
    for folder, label in [
        (RESULTS_DIR, "results"),
        (FIGURES_DIR, "figures")
    ]:
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(
                    path,
                    Path(label) / path.relative_to(folder)
                )

    for name in [
        "run_experiment.py",
        "make_figures.py",
        "requirements.txt"
    ]:
        path = PROJECT_DIR / name
        if path.exists():
            archive.write(path, Path("original_source") / name)

print("Checkpoint created.")
files.download(str(checkpoint))

Checkpoint created.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>